In [1]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, set_seed, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
if torch.cuda.is_available():
    print("CUDA is available. Using GPU.")
else:
    print("CUDA is not available. Using CPU.")

CUDA is not available. Using CPU.


/home/ivanremen/otus/coursework/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [3]:
MODEL_NAME = "fitlemon/bge-m3-ru-ostap"

In [4]:
NUM_LABELS = 3  # 0=negative, 1=neutral, 2=positive

In [5]:
ds = load_dataset('./../datasets/reviews_cleaned')

In [6]:
ds

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text'],
        num_rows: 119048
    })
})

In [7]:
set(ds['train']['language'])

{'kazakh', 'other', 'russian'}

In [8]:
ds_russian = ds.filter(lambda x: x['language'] in ('russian'))

In [9]:
def is_valid(example):
    t = example["combined_text"]
    if t is None:
        return False
    if isinstance(t, float) and np.isnan(t):
        return False
    if isinstance(t, str) and t.strip() == "":
        return False
    return True

ds_russian = ds_russian.filter(is_valid)

In [10]:
ds_russian

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text'],
        num_rows: 111667
    })
})

In [11]:
def calc_sentinent(row):
    sentiment = 1
    if row['rating'] > 3.0:
        sentiment = 2
    elif row['rating'] < 3.0:
        sentiment = 0
    else:
        sentiment = 1
    return {'label': sentiment}

In [12]:
ds_russian = ds_russian.map(calc_sentinent)

In [13]:
ds_russian['train']

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text', 'label'],
    num_rows: 111667
})

In [14]:
ds_russian_splitted = ds_russian['train'].train_test_split(test_size=0.1, seed=42)

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [16]:
def tokenize(batch):
    return tokenizer(batch['combined_text'], truncation=True, max_length=256)

In [17]:
ds_tok = ds_russian_splitted.map(tokenize, batched=True)

In [18]:
ds_tok = ds_tok.remove_columns([c for c in ds_tok["train"].column_names if c not in ("input_ids", "attention_mask", "label")])

In [19]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer,
                                        pad_to_multiple_of=8 if torch.cuda.is_available() else None
                                        )

In [20]:
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {v: k for k, v in id2label.items()}

In [21]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    # ignore_mismatched_sizes=True,  # включите, если ругается на размер классификационной головы
)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: fitlemon/bge-m3-ru-ostap
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }


In [23]:
set_seed(42)

In [24]:
args = TrainingArguments(
    output_dir="./../sentiment_bge_m3_ru_cls",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.05,
    save_steps=1000,
    eval_steps=1000,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    lr_scheduler_type="linear",  # или "cosine"
    gradient_accumulation_steps=2,  # 16*2=32 эффективный train batch
    max_grad_norm=1.0,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [25]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [26]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1 Macro
1000,0.309548,0.150536,0.950748,0.571977
2000,0.279880,0.144084,0.950031,0.616345
3000,0.278255,0.137609,0.952181,0.611873
4000,0.234796,0.141937,0.952718,0.654321
5000,0.243461,0.146862,0.952986,0.659831
6000,0.206106,0.139245,0.956300,0.642787
7000,0.184600,0.154862,0.954688,0.658287
8000,0.151598,0.164099,0.954867,0.645887
9000,0.150256,0.162419,0.955494,0.659176
9423,0.165371,0.158796,0.955673,0.664887


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=9423, training_loss=0.24462118587030368, metrics={'train_runtime': 4655.8998, 'train_samples_per_second': 64.757, 'train_steps_per_second': 2.024, 'total_flos': 7.651735680687456e+16, 'train_loss': 0.24462118587030368, 'epoch': 3.0})

In [28]:
trainer.save_model("./../sentiment_bge_m3_ru_cls/prod")
tokenizer.save_pretrained("./../sentiment_bge_m3_ru_cls/prod")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./../sentiment_bge_m3_ru_cls/prod/tokenizer_config.json',
 './../sentiment_bge_m3_ru_cls/prod/tokenizer.json')

In [35]:
pred = trainer.predict(ds_tok["test"])
val_preds = np.argmax(pred.predictions, axis=-1)

print("Metrics:", pred.metrics)
print(classification_report(ds_tok["test"]["label"], val_preds, target_names=[id2label[i] for i in range(NUM_LABELS)]))

Metrics: {'test_loss': 0.15879599750041962, 'test_accuracy': 0.9556729649861199, 'test_f1_macro': 0.6648869399449866, 'test_runtime': 37.2806, 'test_samples_per_second': 299.54, 'test_steps_per_second': 9.361}
              precision    recall  f1-score   support

    negative       0.71      0.71      0.71       389
     neutral       0.39      0.25      0.30       333
    positive       0.98      0.99      0.98     10445

    accuracy                           0.96     11167
   macro avg       0.69      0.65      0.66     11167
weighted avg       0.95      0.96      0.95     11167



In [36]:
@torch.inference_mode()
def predict_sentiment(texts, max_length=256):
    model.eval()
    device = next(model.parameters()).device
    batch = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    batch = {k: v.to(device) for k, v in batch.items()}
    out = model(**batch)
    probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
    pred_ids = probs.argmax(axis=-1)
    return [
        {
            "label": model.config.id2label[int(i)],
            "scores": {model.config.id2label[j]: float(p[j]) for j in range(probs.shape[1])},
        }
        for i, p in zip(pred_ids, probs)
    ]


In [32]:
print(predict_sentiment(["Товар отличный, доставка быстрая", "Полный ужас, не советую"]))

[{'label': 'positive', 'scores': {'negative': 9.459348802920431e-05, 'neutral': 0.00020221959857735783, 'positive': 0.9997031092643738}}, {'label': 'negative', 'scores': {'negative': 0.9898577332496643, 'neutral': 0.006821109913289547, 'positive': 0.0033210988622158766}}]
